# Vacancies 2026 — Salary Prediction (MAPE)
**Target:** `salary_mean_net` | **Metric:** MAPE | **Models:** LightGBM + CatBoost ensemble

### 1. Imports & Config

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import KFold
from sklearn.preprocessing import OrdinalEncoder
import lightgbm as lgb
from catboost import CatBoostRegressor, Pool
import warnings
warnings.filterwarnings('ignore')

SEED = 42
N_SPLITS = 5
SVD_COMPONENTS = 80
TFIDF_MAX_FEATURES = 5000
np.random.seed(SEED)

### 2. Data Loading & Memory Optimization

In [3]:
def reduce_mem_usage(df: pd.DataFrame) -> pd.DataFrame:
    for col in df.columns:
        col_type = df[col].dtype
        if col_type == object or str(col_type) == 'category':
            continue
        c_min, c_max = df[col].min(), df[col].max()
        if np.issubdtype(col_type, np.integer):
            for dtype in [np.int8, np.int16, np.int32, np.int64]:
                if c_min >= np.iinfo(dtype).min and c_max <= np.iinfo(dtype).max:
                    df[col] = df[col].astype(dtype)
                    break
        elif np.issubdtype(col_type, np.floating):
            if c_min >= np.finfo(np.float32).min and c_max <= np.finfo(np.float32).max:
                df[col] = df[col].astype(np.float32)
    return df


train = pd.read_csv('data/train.csv')
test  = pd.read_csv('data/test_x.csv')


### 3. Text Feature Engineering (TF-IDF + TruncatedSVD)

In [4]:
TEXT_COL = 'lemmaized_wo_stopwords_raw_description'

train[TEXT_COL] = train[TEXT_COL].fillna('')
test[TEXT_COL]  = test[TEXT_COL].fillna('')

tfidf = TfidfVectorizer(
    max_features=TFIDF_MAX_FEATURES,
    sublinear_tf=True,
    min_df=3,
    ngram_range=(1, 2),
    dtype=np.float32,
)
svd = TruncatedSVD(n_components=SVD_COMPONENTS, random_state=SEED)

corpus = pd.concat([train[TEXT_COL], test[TEXT_COL]], ignore_index=True)
tfidf_matrix = tfidf.fit_transform(corpus)
svd_matrix   = svd.fit_transform(tfidf_matrix).astype(np.float32)

svd_cols = [f'svd_{i}' for i in range(SVD_COMPONENTS)]
svd_train = pd.DataFrame(svd_matrix[:len(train)], columns=svd_cols)
svd_test  = pd.DataFrame(svd_matrix[len(train):], columns=svd_cols)

print(f'SVD explained variance: {svd.explained_variance_ratio_.sum():.3f}')

SVD explained variance: 0.224


### 4. Feature Engineering

In [5]:
DROP_COLS = [
    'id', 'salary_mean_net',
    'raw_description', 'raw_branded_description',
    'lemmaized_wo_stopwords_raw_description',
    'lemmaized_wo_stopwords_raw_branded_description',
    'name',  # name_clean is cleaner version
]

# Low-cardinality ordinal: experience has natural order
EXPERIENCE_ORDER = [
    'Нет опыта', 'От 1 года до 3 лет', 'От 3 до 6 лет', 'Более 6 лет'
]

# High-cardinality cols → category dtype for LightGBM native handling
HIGH_CARD_COLS = [
    'unified_address_city', 'unified_address_state', 'unified_address_region',
    'unified_address_country', 'employer_id', 'employer_name',
    'professional_roles_name', 'specializations_profarea_name',
    'employer_industries', 'key_skills_name', 'languages_name',
    'name_clean',
]

# Low-cardinality → OrdinalEncoder (works for both LGB and CB)
LOW_CARD_COLS = [
    'schedule_name', 'employment_name',
    'is_branded_description', 'if_foreign_language',
    'accept_handicapped', 'accept_kids',
]


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # experience ordinal
    exp_map = {v: i for i, v in enumerate(EXPERIENCE_ORDER)}
    df['experience_ord'] = df['experience_name'].map(exp_map).fillna(-1).astype(np.int8)

    # text length features from raw (already loaded)
    # (raw cols dropped later, compute before drop)
    df['desc_word_count'] = df['lemmaized_wo_stopwords_raw_description'].str.split().str.len().fillna(0).astype(np.int16)

    # high-cardinality → category
    for col in HIGH_CARD_COLS:
        if col in df.columns:
            df[col] = df[col].astype('category')

    return df


train = engineer_features(train)
test  = engineer_features(test)

# OrdinalEncoder for low-card cols
oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1, dtype=np.float32)
train[LOW_CARD_COLS] = oe.fit_transform(train[LOW_CARD_COLS].astype(str))
test[LOW_CARD_COLS]  = oe.transform(test[LOW_CARD_COLS].astype(str))

TARGET = 'salary_mean_net'
y = train[TARGET].values.astype(np.float32)

feature_cols = [c for c in train.columns if c not in DROP_COLS and c != 'experience_name']

X_train = pd.concat([train[feature_cols].reset_index(drop=True), svd_train], axis=1)
X_test  = pd.concat([test[[c for c in feature_cols if c in test.columns]].reset_index(drop=True), svd_test], axis=1)

# align columns
X_test = X_test.reindex(columns=X_train.columns)

print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')
print(f'Category cols: {X_train.select_dtypes("category").columns.tolist()}')

X_train: (49051, 100), X_test: (12263, 100)
Category cols: ['employer_name', 'key_skills_name', 'unified_address_city', 'unified_address_state', 'unified_address_region', 'unified_address_country', 'specializations_profarea_name', 'professional_roles_name', 'languages_name', 'name_clean', 'employer_id', 'employer_industries']


### 5. LightGBM — K-Fold OOF Training

In [6]:
lgb_params = {
    'objective': 'mape',
    'metric': 'mape',
    'n_estimators': 3000,
    'learning_rate': 0.03,
    'num_leaves': 127,
    'max_depth': -1,
    'min_child_samples': 20,
    'feature_fraction': 0.7,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'random_state': SEED,
    'n_jobs': -1,
    'verbose': -1,
}

cat_cols_lgb = X_train.select_dtypes('category').columns.tolist()

kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

oof_lgb  = np.zeros(len(X_train), dtype=np.float32)
pred_lgb = np.zeros(len(X_test),  dtype=np.float32)

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train)):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]

    model = lgb.LGBMRegressor(**lgb_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(100, verbose=False),
            lgb.log_evaluation(500),
        ],
        categorical_feature=cat_cols_lgb,
    )

    oof_lgb[val_idx]  = model.predict(X_val)
    pred_lgb         += model.predict(X_test) / N_SPLITS

    fold_mape = np.mean(np.abs((y_val - oof_lgb[val_idx]) / (y_val + 1e-8)))
    print(f'Fold {fold+1} | LGB MAPE: {fold_mape:.4f} | best_iter: {model.best_iteration_}')

lgb_oof_mape = np.mean(np.abs((y - oof_lgb) / (y + 1e-8)))
print(f'\nLightGBM OOF MAPE: {lgb_oof_mape:.4f}')

Fold 1 | LGB MAPE: 0.3348 | best_iter: 46
Fold 2 | LGB MAPE: 0.3420 | best_iter: 34
Fold 3 | LGB MAPE: 0.3384 | best_iter: 39
Fold 4 | LGB MAPE: 0.3407 | best_iter: 37
Fold 5 | LGB MAPE: 0.3328 | best_iter: 43

LightGBM OOF MAPE: 0.3378


### 6. CatBoost — K-Fold OOF Training

In [7]:
cat_cols_cb = X_train.select_dtypes('category').columns.tolist()

# CatBoost requires non-null strings; fillna before astype to avoid NaN passing through Categorical
X_train_cb = X_train.copy()
X_test_cb  = X_test.copy()
for col in cat_cols_cb:
    X_train_cb[col] = X_train_cb[col].cat.add_categories('__NA__').fillna('__NA__').astype(str)
    X_test_cb[col]  = X_test_cb[col].cat.add_categories('__NA__').fillna('__NA__').astype(str)

cb_params = dict(
    loss_function='MAPE',
    eval_metric='MAPE',
    iterations=3000,
    learning_rate=0.03,
    depth=7,
    l2_leaf_reg=3.0,
    random_strength=1.0,
    bagging_temperature=0.5,
    od_type='Iter',
    od_wait=100,
    random_seed=SEED,
    thread_count=-1,
    verbose=500,
)

oof_cb  = np.zeros(len(X_train_cb), dtype=np.float32)
pred_cb = np.zeros(len(X_test_cb),  dtype=np.float32)

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train_cb)):
    X_tr, X_val = X_train_cb.iloc[tr_idx], X_train_cb.iloc[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]

    train_pool = Pool(X_tr, y_tr, cat_features=cat_cols_cb)
    val_pool   = Pool(X_val, y_val, cat_features=cat_cols_cb)

    model = CatBoostRegressor(**cb_params)
    model.fit(train_pool, eval_set=val_pool, use_best_model=True)

    oof_cb[val_idx]  = model.predict(val_pool)
    pred_cb         += model.predict(Pool(X_test_cb, cat_features=cat_cols_cb)) / N_SPLITS

    fold_mape = np.mean(np.abs((y_val - oof_cb[val_idx]) / (y_val + 1e-8)))
    print(f'Fold {fold+1} | CB MAPE: {fold_mape:.4f}')

cb_oof_mape = np.mean(np.abs((y - oof_cb) / (y + 1e-8)))
print(f'\nCatBoost OOF MAPE: {cb_oof_mape:.4f}')

0:	learn: 0.3908842	test: 0.3904453	best: 0.3904453 (0)	total: 92.5ms	remaining: 4m 37s
500:	learn: 0.3738609	test: 0.3772974	best: 0.3772969 (499)	total: 13s	remaining: 1m 4s
1000:	learn: 0.3725407	test: 0.3770421	best: 0.3770393 (987)	total: 25.4s	remaining: 50.7s
1500:	learn: 0.3711111	test: 0.3763516	best: 0.3763475 (1490)	total: 38s	remaining: 37.9s
2000:	learn: 0.3703318	test: 0.3760503	best: 0.3760493 (1999)	total: 50.3s	remaining: 25.1s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.3759266172
bestIteration = 2316

Shrink model to first 2317 iterations.
Fold 1 | CB MAPE: 0.3759
0:	learn: 0.3907655	test: 0.3909227	best: 0.3909227 (0)	total: 27.1ms	remaining: 1m 21s
500:	learn: 0.3705111	test: 0.3729645	best: 0.3729639 (496)	total: 12.5s	remaining: 1m 2s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.3728964901
bestIteration = 559

Shrink model to first 560 iterations.
Fold 2 | CB MAPE: 0.3729
0:	learn: 0.3907007	test: 0.3912550	best: 0

### 7. Ensemble Blending (OOF-weighted)

In [8]:
# Weight inversely proportional to OOF MAPE
w_lgb = 1.0 / lgb_oof_mape
w_cb  = 1.0 / cb_oof_mape
w_sum = w_lgb + w_cb

oof_blend  = (w_lgb * oof_lgb  + w_cb * oof_cb)  / w_sum
pred_blend = (w_lgb * pred_lgb + w_cb * pred_cb) / w_sum

blend_mape = np.mean(np.abs((y - oof_blend) / (y + 1e-8)))
print(f'LGB weight: {w_lgb/w_sum:.3f} | CB weight: {w_cb/w_sum:.3f}')
print(f'Ensemble OOF MAPE: {blend_mape:.4f}')

LGB weight: 0.526 | CB weight: 0.474
Ensemble OOF MAPE: 0.3525


### 8. Post-processing & Submission

In [9]:
final_preds = np.clip(pred_blend, a_min=0, a_max=None)

test_ids = pd.read_csv('data/test_x.csv', usecols=['id'])['id']

submission = pd.DataFrame({
    'ID': test_ids.values,
    'salary_mean_net': final_preds,
})

submission.to_csv('submission.csv', index=False)
print(submission.head(10))
print(f'\nSubmission shape: {submission.shape}')
print(f'Negative preds: {(final_preds < 0).sum()}')
print(f'Pred stats: min={final_preds.min():.0f}, median={np.median(final_preds):.0f}, max={final_preds.max():.0f}')

         ID  salary_mean_net
0  46224201     36816.585938
1  42119402     34156.953125
2  45716401     29469.980469
3  43716203     30636.369141
4  47109602     36947.843750
5  45507200     29364.494141
6  41123802     35239.089844
7  49155602     34332.234375
8  41169603     33391.875000
9  48041400     37267.300781

Submission shape: (12263, 2)
Negative preds: 0
Pred stats: min=23458, median=34308, max=47467
